In [1]:
# @title 1. Install System Dependencies & Clone ComfyUI
import os
from IPython.display import clear_output

# 1. Install system tools
!apt-get update -y && apt-get install -y ffmpeg libgl1-mesa-glx aria2

# 2. Clone ComfyUI
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/Comfy-Org/ComfyUI.git
    print("✅ ComfyUI cloned.")
else:
    print("ℹ️ ComfyUI already exists.")

# 3. Clone ComfyUI Manager (Essential for installing missing nodes)
if not os.path.exists("ComfyUI/custom_nodes/ComfyUI-Manager"):
    %cd ComfyUI/custom_nodes
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git
    %cd ../..
    print("✅ ComfyUI Manager installed.")

# 4. Install Python Requirements
%cd ComfyUI
!pip install -r requirements.txt
!pip install pyngrok  # Needed for the tunnel later

print("✅ Installation complete.")
clear_output()

In [2]:
# @title 2. Apply Storage Hack (Move Models to /tmp)
import os
import shutil

# Define paths
BASE_DIR = "/kaggle/working/ComfyUI"
TEMP_DIR = "/tmp/ComfyUI_Storage" # /tmp usually has ~Disk space on Kaggle

# Create temp directory
os.makedirs(TEMP_DIR, exist_ok=True)

# Function to move folder to temp and symlink back
def setup_storage_link(folder_name):
    src = os.path.join(BASE_DIR, folder_name)
    dst = os.path.join(TEMP_DIR, folder_name)
    
    # If the link already exists, skip
    if os.path.islink(src):
        print(f"ℹ️ {folder_name} is already linked.")
        return

    # If the source folder exists (real folder), move it to temp
    if os.path.exists(src):
        # Create destination if missing
        if not os.path.exists(dst):
            print(f"📦 Moving existing {folder_name} to {dst}...")
            shutil.move(src, dst)
        else:
            # If both exist, merge/delete source (safe for empty default folders)
            print(f"⚠️ {dst} exists. Merging/Replacing {src}...")
            shutil.rmtree(src)
    
    # Create the symlink
    if not os.path.exists(src):
        os.symlink(dst, src)
        print(f"🔗 Linked {src} -> {dst}")

# Apply to heavy folders
setup_storage_link("models")
setup_storage_link("input")
setup_storage_link("output")

print("✅ Storage hack applied successfully.")

📦 Moving existing models to /tmp/ComfyUI_Storage/models...
🔗 Linked /kaggle/working/ComfyUI/models -> /tmp/ComfyUI_Storage/models
📦 Moving existing input to /tmp/ComfyUI_Storage/input...
🔗 Linked /kaggle/working/ComfyUI/input -> /tmp/ComfyUI_Storage/input
📦 Moving existing output to /tmp/ComfyUI_Storage/output...
🔗 Linked /kaggle/working/ComfyUI/output -> /tmp/ComfyUI_Storage/output
✅ Storage hack applied successfully.


In [3]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [4]:
!apt-get update -y
!apt-get install -y aria2 ffmpeg libgl1
!which aria2c

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease               
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease   
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading pac

In [5]:
import os, subprocess

BASE = "/kaggle/working/ComfyUI/models"
DIRS = {
    "text_encoders": f"{BASE}/text_encoders",
    "clip_vision": f"{BASE}/clip_vision",
    "loras": f"{BASE}/loras",
    "diffusion_models": f"{BASE}/diffusion_models",
    "vae": f"{BASE}/vae",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

downloads = [
    # VAE (same as original)
    ("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors",
     f"{DIRS['vae']}/ae.safetensors"),

    # CLIP vision (same as original)
    ("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
     f"{DIRS['clip_vision']}/clip_vision_h.safetensors"),

    # === UNCENSORED Text Encoder (Heretic) ===
    # Recommended: FP8 version (good quality + smaller size)
    ("https://huggingface.co/DreamFast/qwen3-4b-heretic/resolve/main/comfyui/qwen3-4b-heretic_fp8_e4m3fn.safetensors",
     f"{DIRS['text_encoders']}/qwen3-4b-heretic_fp8_e4m3fn.safetensors"),

    # Optional: Full BF16 version (higher quality, larger)
    # ("https://huggingface.co/DreamFast/qwen3-4b-heretic/resolve/main/comfyui/qwen3-4b-heretic.safetensors",
    #  f"{DIRS['text_encoders']}/qwen3-4b-heretic.safetensors"),

    # LoRA (same as original)
    ("https://huggingface.co/mehranpng/lyara-lora/resolve/main/lyara-lora.safetensors",
     f"{DIRS['loras']}/lyara-lora.safetensors"),

    # Diffusion model (same as original)
    ("https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors",
     f"{DIRS['diffusion_models']}/z_image_turbo_bf16.safetensors"),
]

def dl(url, out_path):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1024 * 1024:
        print("✅ Exists:", os.path.basename(out_path))
        return
    cmd = f'aria2c -x 16 -s 16 -c -o "{os.path.basename(out_path)}" -d "{os.path.dirname(out_path)}" "{url}"'
    print("⬇️", os.path.basename(out_path))
    subprocess.run(cmd, shell=True, check=True)

for url, outp in downloads:
    dl(url, outp)

print("\nFolders:")
!ls -lh "{DIRS['text_encoders']}" "{DIRS['clip_vision']}" "{DIRS['loras']}" "{DIRS['diffusion_models']}" "{DIRS['vae']}"

⬇️ ae.safetensors

11/14 12:47:40 [NOTICE] Downloading 1 item(s)

11/14 12:47:40 [NOTICE] CUID#7 - Redirecting to https://huggingface.co/Comfy-Org/Z-Image-Turbo/resolve/main/split_files/vae/ae.safetensors

11/14 12:47:40 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/692651442d28dcd60d675b85/f744f169fc8c7d1c8fcfdfdbd86251d518c801084880373c77e1cf31bcff827e?xip=wKQXSpSzpQQ&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27ae.safetensors%3B+filename%3D%22ae.safetensors%22%3B&user_id=public&Expires=1794664060&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjkyNjUxNDQyZDI4ZGNkNjBkNjc1Yjg1L2Y3NDRmMTY5ZmM4YzdkMWM4ZmNmZGZkYmQ4NjI1MWQ1MThjODAxMDg0ODgwMzczYzc3ZTFjZjMxYmNmZjgyN2VcXD94aXA9d0tRWFNwU3pwUVEmWC1YZXQtQ2FzLVVpZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJnVzZXJfaWQ9cHVibGljIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTQ2NjQwNjB9fX1dfQ__&Signature=MEUCIAG99G6Js

In [ ]:
import subprocess, re, time, os, urllib.request

PORT = 8188
COMFY_DIR = "/kaggle/working/ComfyUI"
COMFY_LOG = "/kaggle/working/comfy.log"
PINGGY_LOG = "/kaggle/working/pinggy.log"

os.system("pkill -f 'python main.py'; pkill -f 'a.pinggy.io'")
time.sleep(1)

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", str(PORT)],
    cwd=COMFY_DIR, stdout=open(COMFY_LOG, "w"), stderr=subprocess.STDOUT,
)

# wait for ComfyUI up
for _ in range(90):
    if comfy.poll() is not None:
        print(open(COMFY_LOG).read()[-3000:])
        raise RuntimeError("ComfyUI died, see log above")
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}", timeout=2)
        print("ComfyUI up")
        break
    except Exception:
        time.sleep(2)
else:
    print(open(COMFY_LOG).read()[-3000:])
    raise RuntimeError("ComfyUI never came up")

pinggy = subprocess.Popen(
    ["ssh", "-T", "-p", "443", "-R0:127.0.0.1:%d" % PORT,
     "-o", "StrictHostKeyChecking=no",
     "-o", "ServerAliveInterval=30",
     "-o", "ServerAliveCountMax=3",
     "a.pinggy.io"],
    stdin=subprocess.DEVNULL,
    stdout=open(PINGGY_LOG, "w"), stderr=subprocess.STDOUT,
)

url = None
for _ in range(60):
    txt = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", open(PINGGY_LOG).read())
    m = re.search(r"https://[^\s]*pinggy\.(?:link|live|io)[^\s]*", txt)
    if m and "dashboard" not in m.group(0):
        url = m.group(0); break
    time.sleep(1)

print("URL:", url) if url else print("No URL. Pinggy log:\n", txt)

while True:   # keep cell alive, tunnel dies if kernel idle-stops
    time.sleep(60)

ComfyUI up
No URL. Pinggy log:
Allocated port 8 for remote forward to 127.0.0.1:8188
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
https://bodwc-34-21-158-112.free.pinggy.net
https://vlwgi-34-21-158-112.run.pinggy-free.link

